# Wasabi Upload — Folder Upload Notebook

| Cell | Purpose |
|------|---------|
| **1** | Install & Imports |
| **2** | Configuration (credentials + region) |
| **3** | All helper functions (run once) |
| **4** | ▶ Upload a single file |
| **5** | ▶ Upload a folder (recursive) |
| **A** | Appendix — Browse bucket contents |

## 1 — Install & Imports

In [1]:
%pip install boto3 tqdm ipywidgets python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import time
import threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from typing import Optional

import boto3
from botocore.config import Config
from botocore.exceptions import ClientError

try:
    from tqdm.auto import tqdm
except ImportError:
    from tqdm import tqdm

from IPython.display import display, HTML

try:
    from dotenv import load_dotenv
    _dotenv_path = Path("..") / ".env"
    load_dotenv(_dotenv_path, override=True)
    print(f"✓ .env loaded from {_dotenv_path.resolve()}")
except Exception as _e:
    print(f"  .env not loaded: {_e}")

✓ .env loaded from /home/taiaburrahman/dataset_manager_pro/.env


## 2 — Configuration

Values are read from the `.env` file automatically.  
Override any value here by assigning directly (overrides the env file).

In [4]:
# ── Wasabi credentials ─────────────────────────────────────────────────────
WASABI_ACCESS_KEY_ID     = os.getenv("WASABI_ACCESS_KEY_ID",     "")
WASABI_SECRET_ACCESS_KEY = os.getenv("WASABI_SECRET_ACCESS_KEY", "")
WASABI_BUCKET            = os.getenv("WASABI_BUCKET",            "")
WASABI_REGION            = os.getenv("WASABI_REGION",            "us-east-1")
WASABI_ENDPOINT          = os.getenv("WASABI_ENDPOINT",          "https://s3.wasabisys.com")

# ── Parallel upload threads ────────────────────────────────────────────────
MAX_WORKERS = 4

print(f"Bucket  : {WASABI_BUCKET}")
print(f"Region  : {WASABI_REGION}")
print(f"Endpoint: {WASABI_ENDPOINT}")

Bucket  : bbvision
Region  : us-east-2
Endpoint: https://s3.us-east-2.wasabisys.com


## 3 — Helper Functions

Run this cell once. It defines all internals used by the upload cells below.

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# ALL HELPER FUNCTIONS — run once before any upload cell
# ══════════════════════════════════════════════════════════════════════════════

def _g(name: str, default: str = "") -> str:
    """Read a config value from module globals → os.environ → default."""
    return globals().get(name) or os.getenv(name, default)

def _bucket() -> str:
    return _g("WASABI_BUCKET", "")

def _workers() -> int:
    return globals().get("MAX_WORKERS", 4)

def get_client() -> boto3.client:
    """Build a boto3 S3 client pointed at the configured Wasabi endpoint."""
    return boto3.client(
        "s3",
        endpoint_url=_g("WASABI_ENDPOINT", "https://s3.wasabisys.com"),
        region_name=_g("WASABI_REGION", "us-east-1"),
        aws_access_key_id=_g("WASABI_ACCESS_KEY_ID"),
        aws_secret_access_key=_g("WASABI_SECRET_ACCESS_KEY"),
        config=Config(signature_version="s3v4",
                      retries={"max_attempts": 3, "mode": "adaptive"}),
    )

def _human_size(n: int) -> str:
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024:
            return f"{n:.1f} {unit}"
        n /= 1024
    return f"{n:.1f} PB"

def _box(lines: list[str], width: int = 68) -> str:
    """Render a bordered summary box."""
    sep   = "─" * (width - 2)
    top   = f"╔{sep}╗"
    bot   = f"╚{sep}╝"
    mid   = f"╠{sep}╣"
    def row(s):
        return f"║  {s:<{width - 4}}║"
    out = [top]
    for line in lines:
        if line == "---":
            out.append(mid)
        else:
            out.append(row(line))
    out.append(bot)
    return "\n".join(out)

def test_connection() -> bool:
    """Verify credentials and bucket access."""
    client = get_client()
    bucket = _bucket()
    try:
        client.head_bucket(Bucket=bucket)
        print(f"✓ Connected — bucket '{bucket}' is accessible")
        return True
    except ClientError as e:
        code = e.response["Error"]["Code"]
        print(f"✗ Connection failed ({code}): {e}")
        return False

def list_bucket(prefix: str = "") -> list[dict]:
    """Return all objects under *prefix* as a list of dicts."""
    client = get_client()
    objects = []
    paginator = client.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=_bucket(), Prefix=prefix):
        for obj in page.get("Contents", []):
            objects.append({
                "key":           obj["Key"],
                "size":          obj["Size"],
                "last_modified": obj["LastModified"].isoformat(),
                "etag":          obj["ETag"].strip('"'),
            })
    return objects


# ── Single file upload ──────────────────────────────────────────────────────

def upload_file(
    local_path: str | Path,
    dest_key: str,
    overwrite: bool = False,
) -> str:
    """
    Upload a single local file to the bucket.

    Parameters
    ----------
    local_path : Local file path, e.g. "/data/file.parquet"
    dest_key   : Full S3 object key, e.g. "datasets/train/file.parquet"
    overwrite  : Re-upload even if the key already exists in the bucket.
    """
    client = get_client()
    src    = Path(local_path)
    key    = dest_key.lstrip("/")

    if not src.exists():
        raise FileNotFoundError(f"Local file not found: {src}")
    if not src.is_file():
        raise ValueError(f"Not a file: {src}")

    file_size = src.stat().st_size

    if not overwrite:
        try:
            client.head_object(Bucket=_bucket(), Key=key)
            print(_box([
                "UPLOAD SKIPPED",
                "---",
                f"File     : {src.name}",
                f"Size     : {_human_size(file_size)}",
                f"Dest key : {key}",
                "---",
                "⏭  Already exists — set OVERWRITE = True to re-upload",
            ]))
            return key
        except ClientError:
            pass

    with tqdm(total=file_size, unit="B", unit_scale=True,
              unit_divisor=1024, desc=src.name,
              bar_format="{l_bar}{bar:30}| {n_fmt}/{total_fmt} [{rate_fmt}]") as bar:
        client.upload_file(str(src), _bucket(), key,
                           Callback=lambda n: bar.update(n))

    print(_box([
        "UPLOAD COMPLETE",
        "---",
        f"File     : {src.name}",
        f"Size     : {_human_size(file_size)}",
        f"Dest key : {key}",
    ]))
    return key


# ── Folder upload ──────────────────────────────────────────────────────────

@dataclass
class _UploadResult:
    source_path:  str
    dest_prefix:  str
    total:        int  = 0
    total_size:   int  = 0
    succeeded:    int  = 0
    skipped_keys: list[str] = field(default_factory=list)
    failed:       list[str] = field(default_factory=list)

    @property
    def skipped(self) -> int:
        return len(self.skipped_keys)

    def print_summary(self):
        lines = [
            "FOLDER UPLOAD SUMMARY",
            "---",
            f"Source folder   : {self.source_path}",
            f"Dest prefix    : {self.dest_prefix}",
            "---",
            f"Total files    : {self.total}  ({_human_size(self.total_size)})",
            f"✓  Uploaded    : {self.succeeded}",
            f"⏭  Skipped     : {self.skipped}",
            f"✗  Failed      : {len(self.failed)}",
        ]
        if self.skipped_keys:
            lines += ["---", "Skipped files (already existed in bucket):"]
            for k in self.skipped_keys[:20]:
                lines.append(f"   · {Path(k).name}")
            if len(self.skipped_keys) > 20:
                lines.append(f"   … and {len(self.skipped_keys) - 20} more")
        if self.failed:
            lines += ["---", "Failed files:"]
            for f in self.failed[:20]:
                lines.append(f"   · {f}")
            if len(self.failed) > 20:
                lines.append(f"   … and {len(self.failed) - 20} more")
        print(_box(lines))


def upload_folder(
    source_path: str | Path,
    dest_prefix: str,
    overwrite: bool = False,
) -> _UploadResult:
    """
    Upload an entire local folder (including all subfolders) to the bucket.

    Parameters
    ----------
    source_path : Local folder path, e.g. "/data/datasets/train/"
    dest_prefix : S3 prefix (virtual folder), e.g. "datasets/train/"
    overwrite   : Re-upload files that already exist in the bucket.
    """
    client   = get_client()
    src_root = Path(source_path).resolve()
    prefix   = dest_prefix.strip("/")
    if prefix:
        prefix += "/"

    if not src_root.exists():
        raise FileNotFoundError(f"Source folder not found: {src_root}")
    if not src_root.is_dir():
        raise ValueError(f"Not a directory: {src_root}")

    # ── Collect all files recursively ───────────────────────────────────────
    all_files = sorted(f for f in src_root.rglob("*") if f.is_file())

    if not all_files:
        print("No files found in source folder.")
        return _UploadResult(source_path=str(src_root), dest_prefix=prefix)

    total_bytes = sum(f.stat().st_size for f in all_files)

    # ── Content breakdown ──────────────────────────────────────────────────
    sub_dirs = {str(f.parent.relative_to(src_root))
                for f in all_files if f.parent != src_root}
    n_root_files = sum(1 for f in all_files if f.parent == src_root)

    print(f"Source folder : {src_root}")
    print(f"Dest prefix   : {prefix}")
    print(f"{'─'*60}")
    print(f"  Files at root level : {n_root_files}")
    print(f"  Sub-folders         : {len(sub_dirs)}")
    if sub_dirs:
        for d in sorted(sub_dirs)[:15]:
            print(f"    └─ {d}")
        if len(sub_dirs) > 15:
            print(f"    … and {len(sub_dirs) - 15} more")
    print(f"  Total files         : {len(all_files)}  ({_human_size(total_bytes)})")
    print(f"{'─'*60}\n")

    # ── Pre-fetch existing keys to skip already-uploaded files ──────────────
    existing_keys = set()
    if not overwrite:
        print("Checking existing objects in bucket …")
        for obj in list_bucket(prefix=prefix):
            existing_keys.add(obj["key"])
        print(f"  Found {len(existing_keys)} existing object(s) under prefix.\n")

    result = _UploadResult(
        source_path=str(src_root), dest_prefix=prefix,
        total=len(all_files), total_size=total_bytes,
    )
    lock = threading.Lock()
    t0   = time.time()

    bytes_bar = tqdm(total=total_bytes, unit="B", unit_scale=True,
                     unit_divisor=1024, desc="Bytes uploaded",
                     bar_format="{l_bar}{bar:30}| {n_fmt}/{total_fmt} [{rate_fmt}]")
    files_bar = tqdm(total=len(all_files), unit="file",
                     desc="Files processed",
                     bar_format="{l_bar}{bar:30}| {n}/{total} files [{elapsed}<{remaining}]")

    def _upload_one(file_path: Path):
        rel_path = file_path.relative_to(src_root)
        key      = prefix + str(rel_path).replace(os.sep, "/")
        fsize    = file_path.stat().st_size

        if key in existing_keys:
            with lock:
                bytes_bar.update(fsize)
            return "skip", key, rel_path

        uploaded = [0]

        def _cb(n):
            uploaded[0] += n
            with lock:
                bytes_bar.update(n)

        try:
            client.upload_file(str(file_path), _bucket(), key, Callback=_cb)
            rem = fsize - uploaded[0]
            if rem > 0:
                with lock:
                    bytes_bar.update(rem)
            return "ok", key, rel_path
        except Exception as exc:
            rem = fsize - uploaded[0]
            if rem > 0:
                with lock:
                    bytes_bar.update(rem)
            return "fail", f"{key} — {exc}", rel_path

    with ThreadPoolExecutor(max_workers=_workers()) as pool:
        futures = {pool.submit(_upload_one, f): f for f in all_files}
        for future in as_completed(futures):
            status, info, rel = future.result()
            with lock:
                files_bar.update(1)
                if status == "ok":
                    result.succeeded += 1
                    files_bar.set_postfix_str(f"✓ {rel}")
                elif status == "skip":
                    result.skipped_keys.append(info)
                    files_bar.set_postfix_str(f"⏭ {rel}")
                else:
                    result.failed.append(info)
                    files_bar.set_postfix_str(f"✗ {rel}")

    bytes_bar.close()
    files_bar.close()

    elapsed = time.time() - t0
    avg_speed = total_bytes / elapsed if elapsed > 0 else 0
    print(f"\nDone in {elapsed:.1f}s — avg {_human_size(avg_speed)}/s\n")
    result.print_summary()
    return result


# ── verify connection on load ──────────────────────────────────────────────────
test_connection()

✓ Connected — bucket 'bbvision' is accessible


True

## 4 — Upload a Single File

Set `LOCAL_PATH` to the local file and `DEST_KEY` to the full S3 key where it should land.

In [ ]:
# ── INPUT ──────────────────────────────────────────────────────────────────
LOCAL_PATH = "/home/taiaburrahman/dataset_manager_pro/data/example.parquet"   # local file
DEST_KEY   = "datasets/example.parquet"                                       # S3 object key in bucket
OVERWRITE  = False   # set True to re-upload even if the key exists
# ───────────────────────────────────────────────────────────────────────────

upload_file(LOCAL_PATH, DEST_KEY, overwrite=OVERWRITE)

## 5 — Upload a Folder (Recursive)

Set `SOURCE_FOLDER` to the local directory and `DEST_PREFIX` to the S3 prefix.  
All files and subfolders are uploaded recursively, preserving the directory structure.

In [6]:
# ── INPUT ──────────────────────────────────────────────────────────────────
SOURCE_FOLDER = "/home/taiaburrahman/dataset_manager_pro/data/projects"   # local folder to upload
DEST_PREFIX   = "datasets/projects"                                       # S3 prefix (virtual folder) in bucket
OVERWRITE     = False   # set True to re-upload existing files
# ───────────────────────────────────────────────────────────────────────────

upload_folder(SOURCE_FOLDER, DEST_PREFIX, overwrite=OVERWRITE)

Source folder : /home/taiaburrahman/dataset_manager_pro/data/projects
Dest prefix   : datasets/projects/
────────────────────────────────────────────────────────────
  Files at root level : 0
  Sub-folders         : 5
    └─ GAID
    └─ GAID/01.raw/fake
    └─ GAID/01.raw/real
    └─ GAID/01.raw/testset
    └─ GAID/08.docs
  Total files         : 46  (189.1 GB)
────────────────────────────────────────────────────────────

Checking existing objects in bucket …
  Found 0 existing object(s) under prefix.



Bytes uploaded:   0%|                              | 0.00/189G [?B/s]

Files processed:   0%|                              | 0/46 files [00:00<?]


Done in 1015.3s — avg 190.7 MB/s

╔──────────────────────────────────────────────────────────────────╗
║  FOLDER UPLOAD SUMMARY                                           ║
╠──────────────────────────────────────────────────────────────────╣
║  Source folder   : /home/taiaburrahman/dataset_manager_pro/data/projects║
║  Dest prefix    : datasets/projects/                             ║
╠──────────────────────────────────────────────────────────────────╣
║  Total files    : 46  (189.1 GB)                                 ║
║  ✓  Uploaded    : 46                                             ║
║  ⏭  Skipped     : 0                                              ║
║  ✗  Failed      : 0                                              ║
╚──────────────────────────────────────────────────────────────────╝


_UploadResult(source_path='/home/taiaburrahman/dataset_manager_pro/data/projects', dest_prefix='datasets/projects/', total=46, total_size=203059385376, succeeded=46, skipped_keys=[], failed=[])

## Appendix — Browse Bucket Contents

Lists objects in the bucket. Set `BROWSE_PREFIX` to narrow to a sub-folder.

In [ ]:
# ── INPUT ──────────────────────────────────────────────────────────────────
BROWSE_PREFIX = ""   # e.g. "datasets/" to narrow the listing
# ───────────────────────────────────────────────────────────────────────────

objects = list_bucket(prefix=BROWSE_PREFIX.lstrip("/"))

if not objects:
    print("No objects found.")
else:
    total_size = sum(o["size"] for o in objects)
    print(f"{'Key':<70} {'Size':>10}  Last Modified")
    print("-" * 100)
    for o in objects:
        print(f"{o['key']:<70} {_human_size(o['size']):>10}  {o['last_modified'][:19]}")
    print("-" * 100)
    print(f"Total: {len(objects)} object(s)  —  {_human_size(total_size)}")